# 06 — Model Evaluation

Evaluación comparativa de **GBR** (HistGradientBoostingRegressor) y **CatBoost** sobre los 5 datasets de features.

**Requiere:** haber ejecutado `05_model_training.ipynb` primero.

Secciones:
1. Comparación de métricas (heatmaps, barras)
2. Diagnóstico de overfitting (CV vs OOF vs Test vs Train)
3. Scatter plots predicho vs real
4. Análisis de residuos
5. Importancia de features
6. Error por estación CTD y por fecha

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

try:
    from catboost import CatBoostRegressor
    CATBOOST_OK = True
except ImportError:
    CATBOOST_OK = False

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
BASE_DIR = Path('..')
RES_DIR  = BASE_DIR / 'results'
MOD_DIR  = RES_DIR / 'models'

MODEL_COLORS = {'GBR': '#2196F3', 'CatBoost': '#4472C4'}
DS_ORDER = ['patch_16px', 'patch_32px', 'patch_64px', 'patch_128px', 'patch_256px']

## 1. Carga de resultados

In [3]:
metrics_df    = pd.read_csv(RES_DIR / 'model_metrics.csv')
test_preds_df = pd.read_csv(RES_DIR / 'test_predictions.csv', parse_dates=['date'])
oof_preds_df  = pd.read_csv(RES_DIR / 'oof_predictions.csv',  parse_dates=['date'])

with open(RES_DIR / 'best_model_info.json') as f:
    best_info = json.load(f)

print(f"Mejores combinaciones (test MAE ↓):")
display(
    metrics_df[['dataset','model','test_mae','test_rmse','test_r2','test_pearson_r',
                'cv_mae','cv_mae_std','oof_mae','n_iters']]
    .sort_values('test_mae')
    .reset_index(drop=True)
    .style.background_gradient(subset=['test_mae','test_rmse'], cmap='RdYlGn_r')
          .background_gradient(subset=['test_r2','test_pearson_r'], cmap='RdYlGn')
          .format(precision=4)
)

print(f"\nMejor combo: {best_info['best_dataset']} × {best_info['best_model']}")

Mejores combinaciones (test MAE ↓):


AttributeError: The '.style' accessor requires jinja2

## 2. Heatmaps de métricas por dataset × modelo

In [ ]:
available_models = metrics_df['model'].unique().tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (metric, label, cmap, fmt) in zip(axes.flatten(), [
    ('test_mae',      'MAE (NTU) — test',  'RdYlGn_r', '.3f'),
    ('test_rmse',     'RMSE (NTU) — test', 'RdYlGn_r', '.3f'),
    ('test_r2',       'R² — test',         'RdYlGn',   '.3f'),
    ('test_pearson_r','Pearson r — test',  'RdYlGn',   '.3f'),
]):
    pivot = (metrics_df.pivot(index='dataset', columns='model', values=metric)
             .reindex(index=[d for d in DS_ORDER if d in metrics_df['dataset'].values],
                      columns=available_models))
    sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap,
                ax=ax, linewidths=0.5, annot_kws={'size': 11})
    ax.set_title(f'{label}')
    ax.set_xlabel('')

plt.suptitle('Métricas test — todos los datasets × modelos', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RES_DIR / 'eval_heatmaps.png', bbox_inches='tight')
plt.show()

## 3. Diagnóstico de overfitting: CV vs OOF vs Test

In [ ]:
fig, axes = plt.subplots(1, len(available_models), figsize=(8 * len(available_models), 5))
if len(available_models) == 1:
    axes = [axes]

for ax, mtype in zip(axes, available_models):
    sub  = metrics_df[metrics_df['model'] == mtype].copy()
    sub  = sub.set_index('dataset').reindex(
               [d for d in DS_ORDER if d in sub.index])
    x    = np.arange(len(sub))
    w    = 0.26

    bars_cv  = ax.bar(x - w, sub['cv_mae'],   width=w, label='CV MAE',
                      color='#5B9BD5', alpha=0.85, edgecolor='white')
    bars_oof = ax.bar(x,     sub['oof_mae'],  width=w, label='OOF MAE',
                      color='#ED7D31', alpha=0.85, edgecolor='white')
    bars_te  = ax.bar(x + w, sub['test_mae'], width=w, label='Test MAE',
                      color='#70AD47', alpha=0.85, edgecolor='white')

    ax.errorbar(x - w, sub['cv_mae'], yerr=sub['cv_mae_std'],
                fmt='none', color='black', capsize=4, linewidth=1.2, zorder=5)

    ax.set_xticks(x)
    ax.set_xticklabels(sub.index, rotation=25, ha='right')
    ax.set_ylabel('MAE (NTU)')
    ax.set_title(f'{mtype}\nCV ± std  |  OOF  |  Test')
    ax.legend()

    # Línea de referencia: si OOF/test >> CV → overfitting
    ax.axhline(sub['cv_mae'].min(), color='gray', lw=0.8, linestyle='--', alpha=0.5)

plt.suptitle('Diagnóstico overfitting: diferencias grandes CV↔Test indican sobreajuste',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RES_DIR / 'eval_overfit_check.png', bbox_inches='tight')
plt.show()

In [ ]:
# Gap relativo: (Test_MAE - CV_MAE) / CV_MAE × 100 — cuánto empeora al pasar a test
metrics_df['gap_pct'] = ((metrics_df['test_mae'] - metrics_df['cv_mae'])
                          / metrics_df['cv_mae'] * 100).round(1)

pivot_gap = (metrics_df.pivot(index='dataset', columns='model', values='gap_pct')
             .reindex(index=[d for d in DS_ORDER if d in metrics_df['dataset'].values],
                      columns=available_models))

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot_gap, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=ax, linewidths=0.5, center=0,
            annot_kws={'size': 11})
ax.set_title('Gap CV → Test (%)\n(Test_MAE − CV_MAE) / CV_MAE × 100\nValores altos → mayor overfitting')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(RES_DIR / 'eval_gap_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Scatter plots: predicho vs real

In [ ]:
# Un scatter por dataset × modelo (mejor combinación de cada dataset)
best_per_ds = (metrics_df.sort_values('test_mae')
               .groupby('dataset').first().reset_index())

n = len(best_per_ds)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
axes = axes.flatten()

for ax, (_, row) in zip(axes, best_per_ds.iterrows()):
    ds, mt = row['dataset'], row['model']
    sub    = test_preds_df[(test_preds_df['dataset'] == ds) &
                            (test_preds_df['model']   == mt)]
    y_t    = sub['turbidity'].values
    y_p    = sub['pred'].values

    mae   = mean_absolute_error(y_t, y_p)
    rmse  = np.sqrt(mean_squared_error(y_t, y_p))
    r2    = r2_score(y_t, y_p)
    pr    = pearsonr(y_t, y_p)[0]

    color = MODEL_COLORS.get(mt, 'steelblue')
    ax.scatter(y_t, y_p, alpha=0.65, s=45, color=color,
               edgecolors='white', linewidth=0.4, zorder=3)
    lim = max(y_t.max(), y_p.max()) * 1.08
    ax.plot([0, lim], [0, lim], 'k--', lw=1.1, alpha=0.5)
    ax.set_xlim(0, lim);  ax.set_ylim(0, lim)
    ax.set_xlabel('Turbidez real (NTU)')
    ax.set_ylabel('Predicha (NTU)')
    ax.set_title(f'{ds}\n{mt}\nMAE={mae:.3f}  R²={r2:.3f}  r={pr:.3f}')
    ax.set_aspect('equal')

for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle('Mejor modelo por dataset — predicho vs real (test set)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RES_DIR / 'eval_scatter_best_per_ds.png', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter para la mejor combinación global: XGBoost vs CatBoost sobre el mejor dataset
best_ds = best_info['best_dataset']
models_avail = test_preds_df[test_preds_df['dataset'] == best_ds]['model'].unique()

fig, axes = plt.subplots(1, len(models_avail), figsize=(6 * len(models_avail), 5))
if len(models_avail) == 1:
    axes = [axes]

for ax, mt in zip(axes, models_avail):
    sub  = test_preds_df[(test_preds_df['dataset'] == best_ds) &
                          (test_preds_df['model']   == mt)]
    y_t, y_p = sub['turbidity'].values, sub['pred'].values
    mae  = mean_absolute_error(y_t, y_p)
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    r2   = r2_score(y_t, y_p)
    pr   = pearsonr(y_t, y_p)[0]
    bias = np.mean(y_p - y_t)

    color = MODEL_COLORS.get(mt, 'steelblue')
    ax.scatter(y_t, y_p, alpha=0.7, s=55, color=color,
               edgecolors='white', linewidth=0.5, zorder=3)
    lim = max(y_t.max(), y_p.max()) * 1.08
    ax.plot([0, lim], [0, lim], 'k--', lw=1.2, alpha=0.5, label='Perfecto')
    ax.set_xlim(0, lim);  ax.set_ylim(0, lim)
    ax.set_xlabel('Turbidez real (NTU)', fontsize=12)
    ax.set_ylabel('Turbidez predicha (NTU)', fontsize=12)
    ax.set_title(
        f'{best_ds} — {mt}\n'
        f'MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}\n'
        f'Pearson r={pr:.3f}  Bias={bias:+.3f}'
    )
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(RES_DIR / 'eval_scatter_best.png', bbox_inches='tight')
plt.show()

## 5. Análisis de residuos

In [ ]:
fig, axes = plt.subplots(2, len(models_avail), figsize=(6 * len(models_avail), 9))
if len(models_avail) == 1:
    axes = axes.reshape(2, 1)

for col, mt in enumerate(models_avail):
    sub      = test_preds_df[(test_preds_df['dataset'] == best_ds) &
                              (test_preds_df['model']   == mt)]
    y_t, y_p = sub['turbidity'].values, sub['pred'].values
    residual = y_p - y_t
    color    = MODEL_COLORS.get(mt, 'steelblue')

    # Distribución de residuos
    ax0 = axes[0, col]
    ax0.hist(residual, bins=20, color=color, edgecolor='white', linewidth=0.5, alpha=0.85)
    ax0.axvline(0, color='black', lw=1.5, linestyle='--')
    ax0.axvline(residual.mean(), color='red', lw=1.2, linestyle=':', label=f'media={residual.mean():+.3f}')
    ax0.set_xlabel('Residuo (NTU)')
    ax0.set_ylabel('Frecuencia')
    ax0.set_title(f'{mt}\nDistribución de residuos')
    ax0.legend(fontsize=9)

    # Residuos vs predicho
    ax1 = axes[1, col]
    ax1.scatter(y_p, residual, alpha=0.65, s=35, color=color,
                edgecolors='white', linewidth=0.3)
    ax1.axhline(0, color='black', lw=1.2, linestyle='--')
    ax1.set_xlabel('Turbidez predicha (NTU)')
    ax1.set_ylabel('Residuo (NTU)')
    ax1.set_title(f'{mt}\nResiduo vs predicho')

plt.suptitle(f'Residuos — dataset: {best_ds}', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RES_DIR / 'eval_residuals.png', bbox_inches='tight')
plt.show()

## 6. Importancia de features

In [ ]:
top_n = 20

fig, axes = plt.subplots(1, len(models_avail), figsize=(8 * len(models_avail), 7))
if len(models_avail) == 1:
    axes = [axes]

feat_cols = pickle.load(open(MOD_DIR / f'{best_ds}__feat_cols.pkl', 'rb'))

for ax, mt in zip(axes, models_avail):
    model_key = f'{best_ds}__{mt}'
    model     = pickle.load(open(MOD_DIR / f'{model_key}.pkl', 'rb'))
    color     = MODEL_COLORS.get(mt, 'steelblue')

    if mt == 'GBR':
        imp = pd.Series(model.feature_importances_, index=feat_cols, name='importance')
    elif mt == 'CatBoost':
        imp = pd.Series(model.get_feature_importance(), index=feat_cols, name='importance')
    else:
        ax.set_visible(False)
        continue

    top = imp.sort_values(ascending=True).tail(top_n)
    top.plot(kind='barh', ax=ax, color=color, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{mt} — Top {top_n} features\n(dataset: {best_ds})')
    ax.set_xlabel('Importancia')

plt.tight_layout()
plt.savefig(RES_DIR / 'eval_feature_importance.png', bbox_inches='tight')
plt.show()

## 7. Error por estación CTD

In [ ]:
fig, axes = plt.subplots(1, len(models_avail), figsize=(8 * len(models_avail), 5))
if len(models_avail) == 1:
    axes = [axes]

for ax, mt in zip(axes, models_avail):
    sub = test_preds_df[(test_preds_df['dataset'] == best_ds) &
                         (test_preds_df['model']   == mt)].copy()
    sub['abs_error'] = (sub['pred'] - sub['turbidity']).abs()
    ctd_err = sub.groupby('ctd').agg(
        mae=('abs_error', 'mean'),
        n=('abs_error', 'count')
    ).sort_values('mae')

    color = MODEL_COLORS.get(mt, 'steelblue')
    bars = ax.barh(ctd_err.index, ctd_err['mae'], color=color,
                   edgecolor='white', linewidth=0.5)
    for bar, n_obs in zip(bars, ctd_err['n']):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'n={n_obs}', va='center', fontsize=8)
    ax.set_xlabel('MAE (NTU)')
    ax.set_title(f'{mt}\nMAE por estación CTD (test set)')

plt.suptitle(f'Error espacial — dataset: {best_ds}', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RES_DIR / 'eval_error_by_ctd.png', bbox_inches='tight')
plt.show()

## 8. Error por fecha de test

In [ ]:
date_rows = []
for (ds, mt, dt), grp in test_preds_df.groupby(['dataset', 'model', 'date']):
    y_t = grp['turbidity'].values
    y_p = grp['pred'].values
    date_rows.append({
        'dataset': ds, 'model': mt, 'date': dt,
        'mae':  mean_absolute_error(y_t, y_p),
        'rmse': np.sqrt(mean_squared_error(y_t, y_p)),
        'n':    len(grp),
    })
date_err = pd.DataFrame(date_rows)

sub_date = date_err[date_err['dataset'] == best_ds]

fig, ax = plt.subplots(figsize=(12, 4))
for mt in models_avail:
    d = sub_date[sub_date['model'] == mt].sort_values('date')
    ax.plot(d['date'].dt.strftime('%Y-%m-%d'), d['mae'],
            'o-', label=mt, color=MODEL_COLORS.get(mt, 'gray'), markersize=6)

ax.set_xlabel('Fecha')
ax.set_ylabel('MAE (NTU)')
ax.set_title(f'MAE por fecha de test — dataset: {best_ds}')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(RES_DIR / 'eval_error_by_date.png', bbox_inches='tight')
plt.show()

## 9. Comparación XGBoost vs CatBoost — escala espacial

In [ ]:
# Evolución de MAE y R² según el diámetro de extracción
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (metric, label, better) in zip(axes, [
    ('test_mae', 'MAE (NTU)', '↓'),
    ('test_r2',  'R²',        '↑'),
]):
    for mt in available_models:
        sub = (metrics_df[metrics_df['model'] == mt]
               .sort_values('diameter_m'))
        ax.plot(sub['diameter_m'], sub[metric],
                'o-', label=mt, color=MODEL_COLORS.get(mt, 'gray'),
                markersize=7, linewidth=2)
        for _, r in sub.iterrows():
            ax.annotate(r['dataset'].replace('patch_','').replace('tabular_200m','tab'),
                        (r['diameter_m'], r[metric]),
                        textcoords='offset points', xytext=(4, 4), fontsize=7)

    ax.set_xlabel('Diámetro de extracción (m)')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs escala espacial ({better} mejor)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RES_DIR / 'eval_scale_effect.png', bbox_inches='tight')
plt.show()

## 10. Resumen ejecutivo

In [ ]:
best_row = metrics_df.loc[metrics_df['test_mae'].idxmin()]

print("═" * 60)
print("RESUMEN DE EVALUACIÓN")
print("═" * 60)
print(f"Mejor combinación  : {best_row['dataset']} × {best_row['model']}")
print(f"Diámetro extracción: {best_row['diameter_m']} m")
print(f"n_iters (final)    : {best_row['n_iters']}")
print()
print(f"Métricas TEST SET:")
print(f"  MAE       = {best_row['test_mae']:.4f} NTU")
print(f"  RMSE      = {best_row['test_rmse']:.4f} NTU")
print(f"  R²        = {best_row['test_r2']:.4f}")
print(f"  Pearson r = {best_row['test_pearson_r']:.4f}")
print(f"  Bias      = {best_row['test_bias']:+.4f} NTU")
print()
print(f"Métricas CV (GroupKFold, k=5):")
print(f"  CV_MAE    = {best_row['cv_mae']:.4f} ± {best_row['cv_mae_std']:.4f} NTU")
print(f"  Gap CV→Test = {best_row['gap_pct']:+.1f}%")
print("═" * 60)

print("\nArchivos de evaluación guardados en results/:")
for fname in ['eval_heatmaps.png', 'eval_overfit_check.png', 'eval_gap_heatmap.png',
              'eval_scatter_best.png', 'eval_residuals.png',
              'eval_feature_importance.png', 'eval_error_by_ctd.png',
              'eval_error_by_date.png', 'eval_scale_effect.png']:
    print(f"  {fname}")